In [0]:
import json
import os
import shutil
import mlflow

mlflow.set_registry_uri("databricks-uc")

MODELO_ORIGINAL = "workspace.fraude_prod.fraude_random_forest"
VERSION_ORIGINAL = "1"
MODELO_SERVING = "workspace.fraude_prod.fraude_random_forest_serving"

URI_ORIGINAL = f"models:/{MODELO_ORIGINAL}/{VERSION_ORIGINAL}"
DIRECTORIO_LOCAL = "/tmp/fraude_rf_serving_patch"

print("Modelo original:", URI_ORIGINAL)
print("Modelo compatible:", MODELO_SERVING)

Modelo original: models:/workspace.fraude_prod.fraude_random_forest/1
Modelo compatible: workspace.fraude_prod.fraude_random_forest_serving


In [0]:
shutil.rmtree(DIRECTORIO_LOCAL, ignore_errors=True)
os.makedirs(DIRECTORIO_LOCAL, exist_ok=True)

modelo_local = mlflow.artifacts.download_artifacts(
    artifact_uri=URI_ORIGINAL,
    dst_path=DIRECTORIO_LOCAL
)

archivos_con_prune_tree = []

for raiz, _, archivos in os.walk(modelo_local):
    for archivo in archivos:
        ruta = os.path.join(raiz, archivo)

        try:
            with open(ruta, "r", encoding="utf-8") as f:
                contenido = f.read()

            if "pruneTree" in contenido:
                archivos_con_prune_tree.append(ruta)
        except (UnicodeDecodeError, PermissionError, IsADirectoryError):
            pass

print("Modelo descargado en:", modelo_local)
print("Archivos con pruneTree:", len(archivos_con_prune_tree))

for ruta in archivos_con_prune_tree:
    print(ruta)

Modelo descargado en: /tmp/fraude_rf_serving_patch/
Archivos con pruneTree: 1
/tmp/fraude_rf_serving_patch/sparkml/stages/3_RandomForestClassifier_73ce9d86771d/metadata/part-00000-tid-2575992076005951595-cb84b66d-6c3a-4271-baac-7106cd42d91c-546-1-c000.txt


In [0]:
def eliminar_prune_tree(objeto):
    if isinstance(objeto, dict):
        return {
            clave: eliminar_prune_tree(valor)
            for clave, valor in objeto.items()
            if clave != "pruneTree"
        }

    if isinstance(objeto, list):
        return [eliminar_prune_tree(valor) for valor in objeto]

    return objeto


archivos_modificados = 0

for ruta in archivos_con_prune_tree:
    with open(ruta, "r", encoding="utf-8") as f:
        contenido_original = f.read()

    try:
        metadata = json.loads(contenido_original)
    except json.JSONDecodeError:
        print("No se modificó porque no es JSON:", ruta)
        continue

    metadata_compatible = eliminar_prune_tree(metadata)

    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(metadata_compatible, f, separators=(",", ":"))

    archivos_modificados += 1
    print("Modificado:", ruta)

print("Total de archivos modificados:", archivos_modificados)

if archivos_modificados == 0:
    raise RuntimeError(
        "No se modificó ningún archivo. Detén el proceso y revisa la celda anterior."
    )

Modificado: /tmp/fraude_rf_serving_patch/sparkml/stages/3_RandomForestClassifier_73ce9d86771d/metadata/part-00000-tid-2575992076005951595-cb84b66d-6c3a-4271-baac-7106cd42d91c-546-1-c000.txt
Total de archivos modificados: 1


In [0]:
referencias_restantes = []

for raiz, _, archivos in os.walk(modelo_local):
    for archivo in archivos:
        ruta = os.path.join(raiz, archivo)

        try:
            with open(ruta, "r", encoding="utf-8") as f:
                if "pruneTree" in f.read():
                    referencias_restantes.append(ruta)
        except (UnicodeDecodeError, PermissionError, IsADirectoryError):
            pass

print("Referencias restantes:", len(referencias_restantes))

if referencias_restantes:
    for ruta in referencias_restantes:
        print(ruta)
    raise RuntimeError("Todavía existen referencias a pruneTree.")

print("Copia compatible preparada correctamente.")

Referencias restantes: 0
Copia compatible preparada correctamente.


In [0]:
import mlflow.spark

RUTA_TEMPORAL_UC = (
    "/Volumes/workspace/fraude_prod/mlflow_tmp/spark_models"
)

os.environ["MLFLOW_DFS_TMP"] = RUTA_TEMPORAL_UC

modelo_original_spark = mlflow.spark.load_model(
    URI_ORIGINAL,
    dfs_tmpdir=RUTA_TEMPORAL_UC
)

modelo_compatible_spark = mlflow.spark.load_model(
    modelo_local,
    dfs_tmpdir=RUTA_TEMPORAL_UC
)

print("Modelo original cargado")
print("Copia compatible cargada")

{"ts": "2026-09-19 14:20:28.102", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzUwMTE1NzQ3MjE5NzY2MhABIAEyJDAxYTBiYTA3LTBmZWEtN2Y0Yy1hZmUxLTQyNWQyNjI1ODE1NzokNjI2ZjAwN2YtMDhjMi0zMDUzLWJlYjctOGU4YzhkODI2YzMySgsI3LK61QYQgO3mQVABWAFgAWiemejlmsWjDQ==.", "context": {}}
{"ts": "2026-09-19 14:20:28.102", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzUwMTE1NzQ3MjE5NzY2MhABIAEyJDAxYTBiYTA3LTBmZWEtN2Y0Yy1hZmUxLTQyNWQyNjI1ODE1NzokNjI2ZjAwN2YtMDhjMi0zMDUzLWJlYjctOGU4YzhkODI2YzMySgsI3LK61QYQgO3mQVABWAFgAWiemejlmsWjDQ==.", "context": {}}
{"ts": "2026-09-19 14:20:28.102", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzUwMTE1NzQ3MjE5NzY2MhABIAEyJDAxYTBiYTA3LTBmZWEtN2Y0Yy1hZmUxLTQyNWQyNjI1ODE1NzokNjI2ZjAwN2YtMDhjMi0zMDUzLWJlYjctOGU4YzhkODI2YzMy

Modelo original cargado
Copia compatible cargada


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

FEATURE_COLS = [
    "edad_ingreso_asegurado",
    "edad_actual_asegurado",
    "vigencia_poliza",
    "vigencia_certificado",
    "Sum_Valor_Reservas_Inicial",
    "A_o",
    "dias_siniestro_a_apertura",
    "dias_siniestro_a_notificacion",
    "dias_notificacion_a_apertura",
    "dias_vigencia_cert_a_siniestro",
    "dias_vigencia_pol_a_siniestro",
    "dias_expedicion_a_siniestro",
    "delta_edad",
    "edad_actual_invalida",
    "log_reserva_inicial",
    "reserva_inicial_es_cero"
]

TABLA_ENTRADA = (
    "workspace.fraude_prod.reclamaciones_entrada_demo"
)

entrada = spark.table(TABLA_ENTRADA)

date_cols = [
    "Fecha_Primera_Vigencia_Cert",
    "fecha_primera_vigencia_pol",
    "FEXPEDICION",
    "FSINIESTRO",
    "F_Notificacion",
    "Fecha_Recepcion",
    "Fecha_Apertura"
]

preparada = entrada

for columna in date_cols:
    if columna in preparada.columns:
        preparada = preparada.withColumn(
            columna,
            F.to_date(F.col(columna))
        )

preparada = (
    preparada
    .withColumn(
        "dias_siniestro_a_apertura",
        F.datediff("Fecha_Apertura", "FSINIESTRO")
    )
    .withColumn(
        "dias_siniestro_a_notificacion",
        F.datediff("F_Notificacion", "FSINIESTRO")
    )
    .withColumn(
        "dias_notificacion_a_apertura",
        F.datediff("Fecha_Apertura", "F_Notificacion")
    )
    .withColumn(
        "dias_vigencia_cert_a_siniestro",
        F.datediff(
            "FSINIESTRO",
            "Fecha_Primera_Vigencia_Cert"
        )
    )
    .withColumn(
        "dias_vigencia_pol_a_siniestro",
        F.datediff(
            "FSINIESTRO",
            "fecha_primera_vigencia_pol"
        )
    )
    .withColumn(
        "dias_expedicion_a_siniestro",
        F.datediff("FSINIESTRO", "FEXPEDICION")
    )
    .withColumn(
        "delta_edad",
        F.col("edad_actual_asegurado")
        - F.col("edad_ingreso_asegurado")
    )
    .withColumn(
        "edad_actual_invalida",
        F.when(
            F.col("edad_actual_asegurado") < 0,
            1.0
        ).otherwise(0.0)
    )
    .withColumn(
        "log_reserva_inicial",
        F.log1p(
            F.greatest(
                F.col("Sum_Valor_Reservas_Inicial"),
                F.lit(0)
            )
        )
    )
    .withColumn(
        "reserva_inicial_es_cero",
        F.when(
            F.col("Sum_Valor_Reservas_Inicial") == 0,
            1.0
        ).otherwise(0.0)
    )
)

columnas_faltantes = [
    columna
    for columna in FEATURE_COLS
    if columna not in preparada.columns
]

print("Columnas faltantes:", columnas_faltantes)

if columnas_faltantes:
    raise RuntimeError(
        f"Faltan columnas: {columnas_faltantes}"
    )

muestra = preparada.select(
    "id_reclamacion",
    *[
        F.col(c).cast("double").alias(c)
        for c in FEATURE_COLS
    ]
).limit(100)

print("Registros de prueba:", muestra.count())

Columnas faltantes: []
Registros de prueba: 100


In [0]:
pred_original = (
    modelo_original_spark
    .transform(muestra)
    .select(
        "id_reclamacion",
        F.col("prediction").alias("pred_original"),
        vector_to_array("probability")[1].alias("score_original")
    )
)

pred_compatible = (
    modelo_compatible_spark
    .transform(muestra)
    .select(
        "id_reclamacion",
        F.col("prediction").alias("pred_compatible"),
        vector_to_array("probability")[1].alias("score_compatible")
    )
)

comparacion = (
    pred_original
    .join(pred_compatible, "id_reclamacion", "inner")
    .withColumn(
        "diferencia_score",
        F.abs(
            F.col("score_original")
            - F.col("score_compatible")
        )
    )
)

resultado = comparacion.agg(
    F.count("*").alias("registros_comparados"),
    F.sum(
        F.when(
            F.col("pred_original") != F.col("pred_compatible"),
            1
        ).otherwise(0)
    ).alias("predicciones_diferentes"),
    F.max("diferencia_score").alias("max_diferencia_score")
)

display(resultado)

registros_comparados,predicciones_diferentes,max_diferencia_score
100,0,0.0


In [0]:
from mlflow.tracking import MlflowClient

with mlflow.start_run(
    run_name="preparar_random_forest_para_serving"
) as run:
    mlflow.log_artifacts(
        modelo_local,
        artifact_path="modelo_serving"
    )

    run_id_serving = run.info.run_id

URI_ARTEFACTO_SERVING = (
    f"runs:/{run_id_serving}/modelo_serving"
)

print("Run ID:", run_id_serving)
print("URI del artefacto:", URI_ARTEFACTO_SERVING)

Run ID: 70a116ffc90645fa99b770517ddd453a
URI del artefacto: runs:/70a116ffc90645fa99b770517ddd453a/modelo_serving


In [0]:
version_serving = mlflow.register_model(
    model_uri=URI_ARTEFACTO_SERVING,
    name=MODELO_SERVING
)

print("Modelo registrado:", version_serving.name)
print("Versión:", version_serving.version)
print("Estado inicial:", version_serving.status)

Successfully registered model 'workspace.fraude_prod.fraude_random_forest_serving'.


Uploading artifacts:   0%|          | 0/47 [00:00<?, ?it/s]

Modelo registrado: workspace.fraude_prod.fraude_random_forest_serving
Versión: 1
Estado inicial: READY


🔗 Created version '1' of model 'workspace.fraude_prod.fraude_random_forest_serving': https://<workspace-databricks>/explore/data/models/workspace/fraude_prod/fraude_random_forest_serving/version/1?o=<workspace-id>


In [0]:
cliente = MlflowClient()

cliente.set_registered_model_alias(
    name=MODELO_SERVING,
    alias="Serving",
    version=version_serving.version
)



cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving.version,
    key="modelo_origen",
    value=f"{MODELO_ORIGINAL}:{VERSION_ORIGINAL}"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving.version,
    key="ajuste_compatibilidad",
    value="remove_pruneTree_metadata"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving.version,
    key="equivalencia_validada",
    value="100_rows_zero_difference"
)

print(
    "URI compatible:",
    f"models:/{MODELO_SERVING}@Serving"
)

URI compatible: models:/workspace.fraude_prod.fraude_random_forest_serving@Serving


In [0]:
import shutil

DIRECTORIO_VERIFICACION = (
    "/tmp/verificar_modelo_serving_registrado"
)

shutil.rmtree(
    DIRECTORIO_VERIFICACION,
    ignore_errors=True
)

os.makedirs(
    DIRECTORIO_VERIFICACION,
    exist_ok=True
)

URI_REGISTRADA = (
    "models:/workspace.fraude_prod."
    "fraude_random_forest_serving/1"
)

modelo_registrado_local = (
    mlflow.artifacts.download_artifacts(
        artifact_uri=URI_REGISTRADA,
        dst_path=DIRECTORIO_VERIFICACION
    )
)

referencias_registradas = []

for raiz, _, archivos in os.walk(modelo_registrado_local):
    for archivo in archivos:
        ruta = os.path.join(raiz, archivo)

        try:
            with open(ruta, "r", encoding="utf-8") as f:
                if "pruneTree" in f.read():
                    referencias_registradas.append(ruta)
        except (
            UnicodeDecodeError,
            PermissionError,
            IsADirectoryError
        ):
            pass

print("URI inspeccionada:", URI_REGISTRADA)
print("Directorio:", modelo_registrado_local)
print(
    "Referencias pruneTree:",
    len(referencias_registradas)
)

for ruta in referencias_registradas:
    print(ruta)

URI inspeccionada: models:/workspace.fraude_prod.fraude_random_forest_serving/1
Directorio: /tmp/verificar_modelo_serving_registrado/
Referencias pruneTree: 0


In [0]:
RUTA_SPARK_CORREGIDA = os.path.join(
    modelo_registrado_local,
    "sparkml"
)

print("Ruta Spark:", RUTA_SPARK_CORREGIDA)
print(
    "Existe:",
    os.path.isdir(RUTA_SPARK_CORREGIDA)
)

if not os.path.isdir(RUTA_SPARK_CORREGIDA):
    raise RuntimeError(
        "No se encontró la carpeta sparkml corregida."
    )

Ruta Spark: /tmp/verificar_modelo_serving_registrado/sparkml
Existe: True


In [0]:
firma_original = mlflow.models.get_model_info(
    URI_ORIGINAL
).signature

with mlflow.start_run(
    run_name="empaquetado_pyfunc_serving_limpio"
):
    modelo_pyfunc = mlflow.pyfunc.log_model(
        artifact_path="modelo_serving_limpio",
        loader_module="mlflow.spark",
        data_path=RUTA_SPARK_CORREGIDA,
        signature=firma_original,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            "pyspark==4.1.0",
            "pandas>=2.0.0"
        ]
    )

print("URI limpia:", modelo_pyfunc.model_uri)

2026/09/19 14:47:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://<workspace-databricks>/ml/experiments/3501157472197662/models/m-45fbfd2c512e4e948a86e0a4dabcd2d0?o=<workspace-id>
/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3285: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


URI limpia: models:/m-45fbfd2c512e4e948a86e0a4dabcd2d0


In [0]:
version_serving_v2 = mlflow.register_model(
    model_uri=modelo_pyfunc.model_uri,
    name=MODELO_SERVING
)

cliente.set_registered_model_alias(
    name=MODELO_SERVING,
    alias="Serving",
    version=version_serving_v2.version
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving_v2.version,
    key="modelo_origen",
    value=f"{MODELO_ORIGINAL}:{VERSION_ORIGINAL}"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving_v2.version,
    key="equivalencia_validada",
    value="100_rows_zero_difference"
)

print("Nueva versión:", version_serving_v2.version)
print(
    "URI:",
    f"models:/{MODELO_SERVING}/"
    f"{version_serving_v2.version}"
)

Registered model 'workspace.fraude_prod.fraude_random_forest_serving' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/45 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.fraude_prod.fraude_random_forest_serving': https://<workspace-databricks>/explore/data/models/workspace/fraude_prod/fraude_random_forest_serving/version/2?o=<workspace-id>


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4884863864419864>, line 6
      1 version_serving_v2 = mlflow.register_model(
      2     model_uri=modelo_pyfunc.model_uri,
      3     name=MODELO_SERVING
      4 )
----> 6 cliente.set_registered_model_alias(
      7     name=MODELO_SERVING,
      8     alias="Serving",
      9     version=version_serving_v2.version
     10 )
     12 cliente.set_model_version_tag(
     13     name=MODELO_SERVING,
     14     version=version_serving_v2.version,
     15     key="modelo_origen",
     16     value=f"{MODELO_ORIGINAL}:{VERSION_ORIGINAL}"
     17 )
     19 cliente.set_model_version_tag(
     20     name=MODELO_SERVING,
     21     version=version_serving_v2.version,
     22     key="equivalencia_validada",
     23     value="100_rows_zero_difference"
     24 )

NameError: name 'cliente' is not defined

In [0]:
from mlflow.tracking import MlflowClient

cliente = MlflowClient()
VERSION_SERVING = "2"

cliente.set_registered_model_alias(
    name=MODELO_SERVING,
    alias="Serving",
    version=VERSION_SERVING
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=VERSION_SERVING,
    key="modelo_origen",
    value=f"{MODELO_ORIGINAL}:{VERSION_ORIGINAL}"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=VERSION_SERVING,
    key="equivalencia_validada",
    value="100_rows_zero_difference"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=VERSION_SERVING,
    key="tipo_empaquetado",
    value="fresh_mlflow_pyfunc_for_serving"
)

version_alias = cliente.get_model_version_by_alias(
    name=MODELO_SERVING,
    alias="Serving"
)

print("Modelo:", version_alias.name)
print("Versión con alias Serving:", version_alias.version)

Modelo: workspace.fraude_prod.fraude_random_forest_serving
Versión con alias Serving: 2


In [0]:
ARCHIVO_MLMODEL = os.path.join(
    modelo_registrado_local,
    "MLmodel"
)

print("Directorio completo:", modelo_registrado_local)
print(
    "Existe MLmodel:",
    os.path.isfile(ARCHIVO_MLMODEL)
)

if not os.path.isfile(ARCHIVO_MLMODEL):
    raise RuntimeError(
        "El directorio no contiene MLmodel."
    )

Directorio completo: /tmp/verificar_modelo_serving_registrado/
Existe MLmodel: True


In [0]:
firma_original = mlflow.models.get_model_info(
    URI_ORIGINAL
).signature

with mlflow.start_run(
    run_name="wrapper_serving_completo"
):
    modelo_pyfunc_v3 = mlflow.pyfunc.log_model(
        artifact_path="modelo_serving_completo",
        loader_module="mlflow.spark",
        data_path=modelo_registrado_local,
        signature=firma_original,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            "pyspark==4.1.0",
            "pandas>=2.0.0"
        ]
    )

print("URI del paquete:", modelo_pyfunc_v3.model_uri)

2026/09/19 15:06:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://<workspace-databricks>/ml/experiments/3501157472197662/models/m-b6029e35a2354371834595568331b38c?o=<workspace-id>
/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3285: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


URI del paquete: models:/m-b6029e35a2354371834595568331b38c


In [0]:
DIRECTORIO_VALIDACION_V3 = (
    "/tmp/validar_paquete_serving_v3"
)

shutil.rmtree(
    DIRECTORIO_VALIDACION_V3,
    ignore_errors=True
)

paquete_v3_local = (
    mlflow.artifacts.download_artifacts(
        artifact_uri=modelo_pyfunc_v3.model_uri,
        dst_path=DIRECTORIO_VALIDACION_V3
    )
)

mlmodel_interno = os.path.join(
    paquete_v3_local,
    "data",
    "MLmodel"
)

print(
    "MLmodel interno disponible:",
    os.path.isfile(mlmodel_interno)
)

referencias_v3 = []

for raiz, _, archivos in os.walk(paquete_v3_local):
    for archivo in archivos:
        ruta = os.path.join(raiz, archivo)

        try:
            with open(ruta, "r", encoding="utf-8") as f:
                if "pruneTree" in f.read():
                    referencias_v3.append(ruta)
        except (
            UnicodeDecodeError,
            PermissionError,
            IsADirectoryError
        ):
            pass

print("Referencias pruneTree:", len(referencias_v3))

MLmodel interno disponible: False
Referencias pruneTree: 0


In [0]:
import yaml

mlmodel_externo = os.path.join(
    paquete_v3_local,
    "MLmodel"
)

with open(
    mlmodel_externo,
    "r",
    encoding="utf-8"
) as f:
    configuracion_mlflow = yaml.safe_load(f)

ruta_data_relativa = (
    configuracion_mlflow
    ["flavors"]
    ["python_function"]
    ["data"]
)

ruta_data_real = os.path.join(
    paquete_v3_local,
    ruta_data_relativa
)

mlmodel_interno_real = os.path.join(
    ruta_data_real,
    "MLmodel"
)

print("Data declarada:", ruta_data_relativa)
print("Ruta real:", ruta_data_real)
print(
    "MLmodel interno real:",
    os.path.isfile(mlmodel_interno_real)
)

print("\nArchivos MLmodel encontrados:")

for raiz, _, archivos in os.walk(paquete_v3_local):
    for archivo in archivos:
        if archivo == "MLmodel":
            print(
                os.path.relpath(
                    os.path.join(raiz, archivo),
                    paquete_v3_local
                )
            )

Data declarada: data/verificar_modelo_serving_registrado
Ruta real: /tmp/validar_paquete_serving_v3/data/verificar_modelo_serving_registrado
MLmodel interno real: True

Archivos MLmodel encontrados:
MLmodel
data/verificar_modelo_serving_registrado/MLmodel
data/verificar_modelo_serving_registrado/metadata/MLmodel
metadata/MLmodel


In [0]:
from mlflow.tracking import MlflowClient

version_serving_v3 = mlflow.register_model(
    model_uri=modelo_pyfunc_v3.model_uri,
    name=MODELO_SERVING
)

cliente = MlflowClient()

cliente.set_registered_model_alias(
    name=MODELO_SERVING,
    alias="Serving",
    version=version_serving_v3.version
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving_v3.version,
    key="modelo_origen",
    value=f"{MODELO_ORIGINAL}:{VERSION_ORIGINAL}"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving_v3.version,
    key="equivalencia_validada",
    value="100_rows_zero_difference"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_serving_v3.version,
    key="tipo_empaquetado",
    value="nested_complete_mlflow_model"
)

print("Versión creada:", version_serving_v3.version)
print(
    "Alias Serving:",
    cliente.get_model_version_by_alias(
        name=MODELO_SERVING,
        alias="Serving"
    ).version
)

Registered model 'workspace.fraude_prod.fraude_random_forest_serving' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/56 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.fraude_prod.fraude_random_forest_serving': https://<workspace-databricks>/explore/data/models/workspace/fraude_prod/fraude_random_forest_serving/version/3?o=<workspace-id>


Versión creada: 3
Alias Serving: 3


In [0]:
import json
import os
import shutil
import yaml
import mlflow

from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")

MODELO_SERVING = (
    "workspace.fraude_prod."
    "fraude_random_forest_serving"
)

URI_VERSION_1 = f"models:/{MODELO_SERVING}/1"
URI_VERSION_2 = f"models:/{MODELO_SERVING}/2"

BASE_FINAL = "/tmp/fraude_serving_final"
V1_LOCAL = os.path.join(BASE_FINAL, "v1")
V2_LOCAL = os.path.join(BASE_FINAL, "v2")

shutil.rmtree(BASE_FINAL, ignore_errors=True)
os.makedirs(V1_LOCAL, exist_ok=True)
os.makedirs(V2_LOCAL, exist_ok=True)

# V1 contiene el MLmodel Spark limpio.
modelo_v1_local = mlflow.artifacts.download_artifacts(
    artifact_uri=URI_VERSION_1,
    dst_path=V1_LOCAL
)

# V2 contiene el wrapper con data/sparkml.
paquete_final = mlflow.artifacts.download_artifacts(
    artifact_uri=URI_VERSION_2,
    dst_path=V2_LOCAL
)

with open(
    os.path.join(paquete_final, "MLmodel"),
    "r",
    encoding="utf-8"
) as f:
    config_externa = yaml.safe_load(f)

data_relativa = (
    config_externa
    ["flavors"]
    ["python_function"]
    ["data"]
)

ruta_sparkml = os.path.join(
    paquete_final,
    data_relativa
)

mlmodel_origen = os.path.join(
    modelo_v1_local,
    "MLmodel"
)

mlmodel_destino = os.path.join(
    os.path.dirname(ruta_sparkml),
    "MLmodel"
)

shutil.copy2(
    mlmodel_origen,
    mlmodel_destino
)

print("Data Spark declarada:", data_relativa)
print(
    "Carpeta Spark disponible:",
    os.path.isdir(ruta_sparkml)
)
print(
    "MLmodel hermano disponible:",
    os.path.isfile(mlmodel_destino)
)

referencias_prune_tree = []

for raiz, _, archivos in os.walk(paquete_final):
    for archivo in archivos:
        ruta = os.path.join(raiz, archivo)

        try:
            with open(ruta, "r", encoding="utf-8") as f:
                if "pruneTree" in f.read():
                    referencias_prune_tree.append(ruta)
        except (
            UnicodeDecodeError,
            PermissionError,
            IsADirectoryError
        ):
            pass

print(
    "Referencias pruneTree:",
    len(referencias_prune_tree)
)

if not os.path.isdir(ruta_sparkml):
    raise RuntimeError("No existe la carpeta Spark.")

if not os.path.isfile(mlmodel_destino):
    raise RuntimeError("Falta MLmodel junto a sparkml.")

if referencias_prune_tree:
    raise RuntimeError(
        "El paquete todavía contiene pruneTree."
    )

# Validación local antes de registrar.
os.environ["MLFLOW_DFS_TMP"] = (
    "/Volumes/workspace/fraude_prod/"
    "mlflow_tmp/spark_models"
)

modelo_validado = mlflow.pyfunc.load_model(
    paquete_final
)

print("Carga local del paquete: CORRECTA")

# Registrar solamente después de validar.
with mlflow.start_run(
    run_name="spark_serving_layout_final"
) as run:
    mlflow.log_artifacts(
        paquete_final,
        artifact_path="modelo_serving_final"
    )
    uri_final = (
        f"runs:/{run.info.run_id}/"
        "modelo_serving_final"
    )

version_final = mlflow.register_model(
    model_uri=uri_final,
    name=MODELO_SERVING
)

cliente = MlflowClient()

cliente.set_registered_model_alias(
    name=MODELO_SERVING,
    alias="Serving",
    version=version_final.version
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_final.version,
    key="equivalencia_validada",
    value="100_rows_zero_difference"
)

cliente.set_model_version_tag(
    name=MODELO_SERVING,
    version=version_final.version,
    key="compatibilidad_serving",
    value="spark_layout_and_pruneTree_fixed"
)

print("Versión final:", version_final.version)
print("Alias Serving actualizado")

Data Spark declarada: data/sparkml
Carpeta Spark disponible: True
MLmodel hermano disponible: True
Referencias pruneTree: 0


{"ts": "2026-09-19 15:24:46.783", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzUwMTE1NzQ3MjE5NzY2MhABIAEyJDAxYTBiYTQ0LTQxNzAtNzY1Yi1hN2M3LTlhN2Q2OThjZTAwYjokNjI2ZjAwN2YtMDhjMi0zMDUzLWJlYjctOGU4YzhkODI2YzMySgwIhtK61QYQgICS9AFQAVgBYAFonpno5ZrFow0=.", "context": {}}
{"ts": "2026-09-19 15:24:46.783", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzUwMTE1NzQ3MjE5NzY2MhABIAEyJDAxYTBiYTQ0LTQxNzAtNzY1Yi1hN2M3LTlhN2Q2OThjZTAwYjokNjI2ZjAwN2YtMDhjMi0zMDUzLWJlYjctOGU4YzhkODI2YzMySgwIhtK61QYQgICS9AFQAVgBYAFonpno5ZrFow0=.", "context": {}}
{"ts": "2026-09-19 15:24:46.783", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzUwMTE1NzQ3MjE5NzY2MhABIAEyJDAxYTBiYTQ0LTQxNzAtNzY1Yi1hN2M3LTlhN2Q2OThjZTAwYjokNjI2ZjAwN2YtMDhjMi0zMDUzLWJlYjctOGU4YzhkODI2YzMy

Carga local del paquete: CORRECTA


Registered model 'workspace.fraude_prod.fraude_random_forest_serving' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/46 [00:00<?, ?it/s]

🔗 Created version '4' of model 'workspace.fraude_prod.fraude_random_forest_serving': https://<workspace-databricks>/explore/data/models/workspace/fraude_prod/fraude_random_forest_serving/version/4?o=<workspace-id>


Versión final: 4
Alias Serving actualizado
